## Imports

In [1]:
import tensorflow as tf
import os
import shutil
import random
from pathlib import Path
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

## Constants


In [2]:
IMG_SIZE = 256
BATCH_SIZE = 32

In [3]:
import shutil
import random
from pathlib import Path

# Your original dataset folder
SOURCE_DIR = Path(r"D:\GDrive\SeniorProject\datasets\dataset\train")
# New split dataset folder
OUTPUT_DIR = Path(r"D:\GDrive\SeniorProject\datasets\brain_tumor_split")

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

random.seed(42)

# Create split only if it does not already exist
if (OUTPUT_DIR / "train").exists():
    print("Split dataset already exists. Skipping split.")

else:
    print("Creating train/val/test split...")

    for class_folder in SOURCE_DIR.iterdir():
        if class_folder.is_dir():
            images = list(class_folder.glob("*.*"))
            random.shuffle(images)

            total = len(images)
            train_end = int(total * TRAIN_RATIO)
            val_end = train_end + int(total * VAL_RATIO)

            split_data = {
                "train": images[:train_end],
                "val": images[train_end:val_end],
                "test": images[val_end:]
            }

            for split_name, split_images in split_data.items():
                split_class_dir = OUTPUT_DIR / split_name / class_folder.name
                split_class_dir.mkdir(parents=True, exist_ok=True)

                for img in split_images:
                    shutil.copy(img, split_class_dir / img.name)

            print(
                class_folder.name,
                "train:", len(split_data["train"]),
                "val:", len(split_data["val"]),
                "test:", len(split_data["test"])
            )

    print("Done creating split dataset.")

Split dataset already exists. Skipping split.


## Paths

In [4]:
## Paths
PROJECT_BASE_DIR = Path(r"D:\GDrive\SeniorProject\datasets\brain_tumor_split")

TRAIN_DIR = str(PROJECT_BASE_DIR / "train")
VAL_DIR   = str(PROJECT_BASE_DIR / "val")
TEST_DIR  = str(PROJECT_BASE_DIR / "test")

MODELS_DIR = PROJECT_BASE_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True, parents=True)

print("Train:", TRAIN_DIR)
print("Val:", VAL_DIR)
print("Test:", TEST_DIR)

print("Train exists:", os.path.exists(TRAIN_DIR))
print("Val exists:", os.path.exists(VAL_DIR))
print("Test exists:", os.path.exists(TEST_DIR))

print("\nTrain image counts:")
for folder in Path(TRAIN_DIR).iterdir():
    if folder.is_dir():
        count = len(list(folder.glob("*.*")))
        print(f"  {folder.name}: {count} images")

Train: D:\GDrive\SeniorProject\datasets\brain_tumor_split\train
Val: D:\GDrive\SeniorProject\datasets\brain_tumor_split\val
Test: D:\GDrive\SeniorProject\datasets\brain_tumor_split\test
Train exists: True
Val exists: True
Test exists: True

Train image counts:
  glioma_tumor: 1559 images
  meningioma_tumor: 1556 images
  no_tumor: 1258 images
  pituitary_tumor: 1560 images


In [5]:
# # Only merges if dataset2 images are not already present
# MERGE_FLAG = PROJECT_BASE_DIR / ".merged"

# if MERGE_FLAG.exists():
#     print("Already merged, skipping")
# else:
#     print("Merging dataset2...")
#     DATASET2_TRAIN = Path(r"D:\GDrive\SeniorProject\datasets\dataset2\train")
#     DATASET2_TEST  = Path(r"D:\GDrive\SeniorProject\datasets\dataset2\test")

#     def merge_into(source_dir, dest_dir):
#         for class_folder in Path(source_dir).iterdir():
#             if class_folder.is_dir():
#                 dest = Path(dest_dir) / class_folder.name
#                 dest.mkdir(exist_ok=True)
#                 copied = skipped = 0
#                 for img in class_folder.glob("*.*"):
#                     try:
#                         shutil.copy(img, dest / img.name)
#                         copied += 1
#                     except OSError:
#                         skipped += 1
#                 print(f"  {class_folder.name}: copied {copied}, skipped {skipped}")

#     merge_into(DATASET2_TRAIN, TRAIN_DIR)
#     merge_into(DATASET2_TEST, TEST_DIR)
#     MERGE_FLAG.touch()
#     print("Merge done...Flag saved.")

#### Done with Baseline Model training ↓

## Baseline Model

In [6]:
# num_classes = len(labels)

# model_baseline = tf.keras.Sequential([
#     tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
#     tf.keras.layers.MaxPooling2D(2,2),

#     tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
#     tf.keras.layers.MaxPooling2D(2,2),

#     tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
#     tf.keras.layers.MaxPooling2D(2,2),

#     tf.keras.layers.Flatten(),

#     tf.keras.layers.Dense(128, activation='relu'),
#     tf.keras.layers.Dropout(0.5),

#     tf.keras.layers.Dense(num_classes, activation='softmax')
# ])

# model_baseline.summary()

### Compile

In [7]:
# model_baseline.compile(
#     optimizer='adam',
#     loss='categorical_crossentropy',
#     metrics=['accuracy']
# )

### Run Epochs

In [8]:
# import time
# import math

# steps_per_epoch = math.ceil(train_generator.samples / BATCH_SIZE)
# validation_steps = math.ceil(test_generator.samples / BATCH_SIZE)

# start = time.time()
# history_baseline = model_baseline.fit(
#     train_generator,
#     steps_per_epoch=steps_per_epoch,
#     validation_data=test_generator,
#     validation_steps=validation_steps,
#     epochs=EPOCHS
# )
# train_time_sec = time.time() - start

# print("Training time (sec):", round(train_time_sec, 2))

### Results

In [9]:
# baseline_loss, baseline_acc = model_baseline.evaluate(
#     test_generator,
#     steps=validation_steps
# )
# print("Baseline Loss:", baseline_loss)
# print("Baseline Accuracy:", baseline_acc)

### Analysis

In [10]:
# import numpy as np
# from sklearn.metrics import confusion_matrix, classification_report

# # reset generator to start
# test_generator.reset()

# y_prob = model_baseline.predict(test_generator, steps=validation_steps)
# y_pred = np.argmax(y_prob, axis=1)

# y_true = test_generator.classes  # integer labels in directory order
# class_names = list(test_generator.class_indices.keys())

# cm = confusion_matrix(y_true, y_pred)
# print("Confusion Matrix:\n", cm)

# print("\nClassification Report:\n")
# print(classification_report(y_true, y_pred, target_names=class_names))

### Save Model

In [11]:
# os.makedirs(MODELS_DIR, exist_ok=True)

# baseline_model_path = MODELS_DIR / "baseline_cnn.keras"
# model_baseline.save(str(baseline_model_path))
# print(f"Saved to {baseline_model_path}")

# Transfer Model

### Generators

In [12]:
transfer_train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=8,
    width_shift_range=0.04,
    height_shift_range=0.04,
    zoom_range=0.08,
    horizontal_flip=True,
    brightness_range=[0.95, 1.05],
    fill_mode='nearest'
)

transfer_eval_datagen = ImageDataGenerator()

train_gen_tf = transfer_train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_gen_tf = transfer_eval_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen_tf = transfer_eval_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

labels = train_gen_tf.class_indices
print("Class labels:", labels)
print("Generators ready.")

Found 5929 images belonging to 4 classes.
Found 1269 images belonging to 4 classes.
Found 1272 images belonging to 4 classes.
Class labels: {'glioma_tumor': 0, 'meningioma_tumor': 1, 'no_tumor': 2, 'pituitary_tumor': 3}
Generators ready.


## Phase 1

### Model Build + Training
##### Skips training if model is saved already

In [13]:


PHASE1_PATH = MODELS_DIR / "phase1.keras"

if PHASE1_PATH.exists():
    print("Phase 1 already trained. Loading...")
    model_transfer = tf.keras.models.load_model(str(PHASE1_PATH))

else:
    print("Training Phase 1...")

    base_model = EfficientNetB3(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.6)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)

    outputs = Dense(train_gen_tf.num_classes, activation='softmax')(x)

    model_transfer = Model(inputs=base_model.input, outputs=outputs)

    model_transfer.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy']
)

    callbacks_phase1 = [
        EarlyStopping(
            monitor='val_accuracy',
            patience=5,
            restore_best_weights=True,
            mode='max'
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            str(PHASE1_PATH),
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        )
    ]

    history_phase1 = model_transfer.fit(
        train_gen_tf,
        validation_data=val_gen_tf,
        epochs=20,
        callbacks=callbacks_phase1
    )

    model_transfer.save(str(PHASE1_PATH))
    print("Saved: Phase 1")

phase1_loss, phase1_acc = model_transfer.evaluate(val_gen_tf)
print(f"Phase 1 Validation Accuracy: {phase1_acc:.4f}")

Training Phase 1...
43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 28s 1us/step
Epoch 1/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6097 - loss: 0.9879
Epoch 1: val_accuracy improved from None to 0.82348, saving model to D:\GDrive\SeniorProject\datasets\brain_tumor_split\models\phase1.keras

Epoch 1: finished saving model to D:\GDrive\SeniorProject\datasets\brain_tumor_split\models\phase1.keras
186/186 ━━━━━━━━━━━━━━━━━━━━ 795s 4s/step - accuracy: 0.7170 - loss: 0.7996 - val_accuracy: 0.8235 - val_loss: 0.5536 - learning_rate: 0.0010
Epoch 2/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8178 - loss: 0.6133
Epoch 2: val_accuracy improved from 0.82348 to 0.88022, saving model to D:\GDrive\SeniorProject\datasets\brain_tumor_split\models\phase1.keras

Epoch 2: finished saving model to D:\GDrive\SeniorProject\datasets\brain_tumor_split\models\phase1.keras
186/186 ━━━━━━━━━━━━━━━━━━━━ 699s 4s/step - accuracy: 0.8290 - loss: 0.5955 - val_accuracy: 0.8802 - val_loss: 0.4746 - l

## Phase 2

### Fine Tune

In [14]:
PHASE2_PATH = MODELS_DIR / "phase2.keras"

if PHASE2_PATH.exists():
    print("Phase 2 already trained. Loading...")
    model_transfer = tf.keras.models.load_model(str(PHASE2_PATH))

else:
    print("Training Phase 2...")

    # Freeze everything first
    for layer in model_transfer.layers:
        layer.trainable = False

    # Unfreeze last 40 layers (EfficientNet style)
    for layer in model_transfer.layers[-40:]:
        layer.trainable = True

    # Keep BatchNorm frozen (important for stability)
    for layer in model_transfer.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    trainable_count = sum(1 for layer in model_transfer.layers if layer.trainable)
    print("Trainable layers:", trainable_count)

    model_transfer.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy']
    )

    callbacks_phase2 = [
        EarlyStopping(
            monitor='val_accuracy',
            patience=8,
            restore_best_weights=True,
            mode='max'
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.3,
            patience=3,
            min_lr=1e-8,
            verbose=1
        ),
        ModelCheckpoint(
            str(PHASE2_PATH),
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        )
    ]

    history_phase2 = model_transfer.fit(
        train_gen_tf,
        validation_data=val_gen_tf,
        epochs=25,
        callbacks=callbacks_phase2
    )

    model_transfer.save(str(PHASE2_PATH))
    print("Saved: Phase 2")

phase2_loss, phase2_acc = model_transfer.evaluate(val_gen_tf)
print(f"Phase 2 Validation Accuracy: {phase2_acc:.4f}")

Training Phase 2...
Trainable layers: 32
Epoch 1/25
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9232 - loss: 0.4012
Epoch 1: val_accuracy improved from None to 0.94484, saving model to D:\GDrive\SeniorProject\datasets\brain_tumor_split\models\phase2.keras

Epoch 1: finished saving model to D:\GDrive\SeniorProject\datasets\brain_tumor_split\models\phase2.keras
186/186 ━━━━━━━━━━━━━━━━━━━━ 368s 2s/step - accuracy: 0.9317 - loss: 0.3854 - val_accuracy: 0.9448 - val_loss: 0.3385 - learning_rate: 1.0000e-04
Epoch 2/25
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9458 - loss: 0.3521
Epoch 2: val_accuracy improved from 0.94484 to 0.94799, saving model to D:\GDrive\SeniorProject\datasets\brain_tumor_split\models\phase2.keras

Epoch 2: finished saving model to D:\GDrive\SeniorProject\datasets\brain_tumor_split\models\phase2.keras
186/186 ━━━━━━━━━━━━━━━━━━━━ 357s 2s/step - accuracy: 0.9548 - loss: 0.3382 - val_accuracy: 0.9480 - val_loss: 0.3291 - learning_rate: 1.0000e-04
Ep

## Evaluate Improvements

In [15]:
## Final Test Evaluation

import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

test_gen_tf.reset()

test_loss, test_acc = model_transfer.evaluate(test_gen_tf)
print(f"Final Test Accuracy: {test_acc:.4f}")

y_prob = model_transfer.predict(test_gen_tf)
y_pred = np.argmax(y_prob, axis=1)

y_true = test_gen_tf.classes
class_names = list(test_gen_tf.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

print("\n── Summary ──")
print(f"Phase 1 Validation Accuracy: {phase1_acc:.4f}")
print(f"Phase 2 Validation Accuracy: {phase2_acc:.4f}")
print(f"Final Test Accuracy: {test_acc:.4f}")

40/40 ━━━━━━━━━━━━━━━━━━━━ 79s 2s/step - accuracy: 0.9882 - loss: 0.2391
Final Test Accuracy: 0.9882
40/40 ━━━━━━━━━━━━━━━━━━━━ 94s 2s/step
Confusion Matrix:
 [[334   1   0   0]
 [  3 323   2   6]
 [  0   1 268   0]
 [  0   2   0 332]]

Classification Report:
                  precision    recall  f1-score   support

    glioma_tumor       0.99      1.00      0.99       335
meningioma_tumor       0.99      0.97      0.98       334
        no_tumor       0.99      1.00      0.99       269
 pituitary_tumor       0.98      0.99      0.99       334

        accuracy                           0.99      1272
       macro avg       0.99      0.99      0.99      1272
    weighted avg       0.99      0.99      0.99      1272


── Summary ──
Phase 1 Validation Accuracy: 0.9488
Phase 2 Validation Accuracy: 0.9913
Final Test Accuracy: 0.9882
